In [ ]:
import pandas as pd
import plotly.express as px

import sys
sys.path.append('../')
import plotting

# Show fluorescence and temperature profiles over time

In [ ]:
dfs = []

for phase in ('initialMC', 'finalMC'):
    df = pd.read_csv(f"../data/qpcr/Primer3_{phase}_Fvst.csv")
    df = df.loc[:, ~df.columns.str.startswith('X.')]
    df = df.drop(columns=["Text"])
    df = df.rename(columns={"X": "A: time"})
    df.columns = df.columns.str.split(': ').str[1]
    df = df.melt(id_vars=["time"], var_name="sequence", value_name="value")
    df['phase'] = phase
    df['type'] = 'Fluorescence'
    dfs.append(df)

Fvst = pd.concat(dfs, ignore_index=True).reset_index(drop=True)

# for each sequence, introduce a new row at a time of 13*60 with a value of None
for seq in Fvst['sequence'].unique():
        new_row = pd.DataFrame({
            'time': [4 * 60.0],
            'sequence': [seq],
            'value': [None],
            'phase': ['finalMC'],
            'type': ['Fluorescence']
        })
        Fvst = pd.concat([Fvst, new_row], ignore_index=True)

Fvst = Fvst.sort_values(by=['sequence', 'time'])
Fvst

In [ ]:
dfs = []

for phase in ('initialMC', 'finalMC'):
    df = pd.read_csv(f"../data/qpcr/Primer3_{phase}_Tvst.csv")
    df = df.loc[:, ~df.columns.str.startswith('X.')]
    df = df.drop(columns=["Text", "Acquisitions"])
    df = df.rename(columns={"X": "time", "Temperature": "value"})
    df['sequence'] = 'Temperature'
    df['phase'] = phase
    df['type'] = 'Temperature'
    dfs.append(df)

Tvst = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
Tvst

In [ ]:
plotdf = pd.concat([Fvst, Tvst], ignore_index=True).reset_index(drop=True)

# shift the time of all points in phase 'finalMC' by the maximum time of phase 'initialMC'
initialMC_max_time = plotdf.loc[plotdf['phase'] == 'initialMC', 'time'].max()
plotdf.loc[plotdf['phase'] == 'finalMC', 'time'] += initialMC_max_time

# transform time to minutes
plotdf['time'] = plotdf['time'] / 60

plotdf

In [ ]:
colormap = {
    "no motif": "#969696",
    "3 motif": "#de2d26",
    "5 motif": "#3182bd",
    "Temperature": "#636363",
}

fig = px.line(
    plotdf,
    x="time",
    y="value",
    color="sequence",
    facet_row="type",
    facet_row_spacing=0.1,
    color_discrete_map=colormap,
    line_group="phase", # do not connect between the two phases
    render_mode="svg"
)


fig.for_each_trace(lambda t: t.update(connectgaps=False))
fig.for_each_annotation(lambda a: a.update(text=""))

fig.update_layout(
    margin=dict(l=0, r=0, t=12, b=0),
    height=300,
    width=680,
    showlegend=False,
)
fig.update_xaxes(range=[0, 20])
fig.update_xaxes(title_text="Time (min)", row=1, col=1)
fig.update_yaxes(range=[0, 105])
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="Fluorescence", row=2, col=1)
fig = plotting.standardize_plot(fig)
fig.show()
fig.write_image("./SI_figure_primer3_sequences/F_and_T_vs_t.svg",)

# Show the initial and final melting curves

In [ ]:
dfs = []

for phase in ('initialMC', 'finalMC'):
    df = pd.read_csv(f"../data/qpcr/Primer3_{phase}_dFvsT.csv")
    df = df.loc[:, ~df.columns.str.startswith('X.')]
    df = df.rename(columns={"X": "A: temperature"})
    df.columns = df.columns.str.split(': ').str[1]
    df = df.melt(id_vars=["temperature"], var_name="sequence", value_name="value")
    df['phase'] = phase
    dfs.append(df)

dFvsT = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
dFvsT

In [ ]:
colormap = {
    "no motif": "#969696",
    "3 motif": "#de2d26",
    "5 motif": "#3182bd",
}

fig = px.line(
    dFvsT,
    x="temperature",
    y="value",
    color="sequence",
    line_dash="phase",
    line_dash_sequence=["dash", "solid"],
    color_discrete_map=colormap,
    render_mode="svg"
)


fig.update_layout(
    margin=dict(l=0, r=10, t=12, b=0),
    height=200,
    width=680,
    showlegend=False,
)
fig.update_xaxes(title="Temperature (°C)", range=[70, 95])
fig.update_yaxes(title="-dF/dT", range=[0, 8])
fig = plotting.standardize_plot(fig)
fig.show()
fig.write_image("./SI_figure_primer3_sequences/melting_curves_onlyextension.svg",)